# Model selection

The number of boosting iterations, $m_{\text{stop}}$, is the primary
hyperparameter. Each iteration adds the fitted base-learner, multiplied by the
step length $\nu$ (argument `nu`, default 0.1), to the predictor, so
$m_{\text{stop}}$ and $\nu$ together set how far the fit moves from the
offset. In practice $\nu$ is fixed and $m_{\text{stop}}$ is chosen by
cross-validated risk (`cvrisk`), or, for Gaussian and likelihood families, by an
information criterion (`aic`).

## Early stopping as regularization

The boosting path traces a sequence of models of increasing complexity. Under
smooth convex loss and a sufficiently small step, training risk decreases.
Expected test risk often falls and later rises, but a finite-sample curve need
not be smooth, U-shaped, or turn within the budget. Resampling estimates held-out risk along the path, and the selected
$m_{\text{stop}}$ minimizes that estimate.

```{note}
A descent direction guarantees a decrease only for a sufficiently small step;
nonsmooth and calibrated families require checking the recorded path. A flat
or boundary cross-validation curve can reflect limited resolution, fold
variation, a useful plateau, or an insufficient path budget.
See *Flat or monotone risk curves* and *High-variance risk curves*.
```

## Cross-validated risk

`cvrisk` estimates the out-of-sample risk **as a function of $m$** in one pass:

1. Draw $B$ resampling folds, each defining in-bag weights $w^{(b)}$ (out-of-bag
   points get weight 0).
2. For each fold $b$, fit the full path to $m_{\max}$ on the in-bag weights.
   A penalized learner has a target degrees of freedom `df` (a measure of its
   flexibility; `df=4` for a default spline) and a penalty $\lambda$ chosen
   to achieve it. Penalized learners **recompute** their $\lambda$ from `df` under
   $w^{(b)}$ (see [](penalties_df.ipynb)); this recomputation keeps the
   target `df` constant across folds even though the effective sample size
   changes.
3. At every $m$, evaluate the risk on the out-of-bag points:
   $r_b(m) = \sum_i v_i^{(b)}\rho(y_i, \hat f^{[m]}_b) / \sum_i v_i^{(b)}$,
   where the evaluation weight $v_i^{(b)}$ equals the original observation
   weight $w_i$ (1 when no weights are given) on rows whose in-bag weight is
   zero, and 0 on in-bag rows. Bootstrap multiplicities serve only as training
   weights.
4. The cross-validated risk is the fold mean $\bar r(m) = \frac1B \sum_b r_b(m)$;
   the chosen iteration is $\hat m = \argmin_m \bar r(m)$.

```{list-table} Resampling schemes
:header-rows: 1

* - `type`
  - Folds
  - Use case
* - `"bootstrap"`
  - $B$ bootstrap samples (in-bag = multiplicity weights)
  - Python default; stable estimate, slightly optimistic
* - `"kfold"`
  - $B$-fold cross-validation
  - Conservative; each observation is out-of-bag exactly once
* - `"subsampling"`
  - $B$ random splits at a fixed in-bag fraction
  - Flexible; controls the in-bag fraction
```

### Fold membership across the schemes

The three resampling schemes differ in how they assign observations to in-bag
and out-of-bag sets:

- **Bootstrap**: each observation is independently drawn into the in-bag
  set with replacement. On average, 63.2% of observations appear at least
  once in a given fold (the "bootstrap inclusion probability"
  $1 - (1 - 1/n)^n \to 1 - e^{-1}$). The remaining 36.8% are out-of-bag.
  Observations can appear multiple times — the weight is their
  multiplicity.

- **K-fold**: the data is split into $K$ disjoint folds of equal size. Each
  fold uses $K-1$ parts as in-bag (weight 1) and 1 part as out-of-bag
  (weight 0). Every observation is out-of-bag in exactly one fold.

- **Subsampling**: each fold draws a random subset of $\lfloor pn \rfloor$
  observations ($p$ is the argument `prob`, default 0.5). In-bag observations get weight 1;
  out-of-bag observations get weight 0. Observations can be in-bag in
  multiple folds or in none.

```{note}
The penalized learners **recompute** their penalty multiplier $\lambda$
under each fold's weights, because $\mathrm{df}(\lambda)$ depends on $W$. A
fold that down-weights some observations changes the effective sample size
and therefore the $\lambda$ needed to hit a target `df`. This per-fold
re-solve is the dominant cost of `cvrisk` for spline-heavy models — see
[](penalties_df.ipynb).
```

Because the cyclic boosting increments are fixed once fit, the risk at every $m$
is read off a single fitted path per fold, with no refitting per grid point.
The folds are independent and run in parallel in the Rust engine.

### The `risk=` parameter

`boost_control(risk=...)` selects which risk is recorded during fitting:

| `risk` | What is recorded | Impact on path |
|---|---|---|
| `"inbag"` (default) | Risk on the training (in-bag) sample | Standard; the path is fitted to this risk |
| `"oobag"` | Risk on the out-of-bag subsample | Requires `weights` that define in-bag vs OOB; used internally by `cvrisk` |
| `"none"` | No risk recorded (faster, no risk array) | For prediction-only fits |

The recorded `risk_` path has length `mstop + 1` (index 0 is the offset-only
risk). For `"inbag"`, this is the in-sample risk; for `"oobag"`, it is the
out-of-sample risk on the held-out subsample.

## Choosing $m_{\text{stop}}$: practical guidance

### Choosing the number of folds

- **Bootstrap**: $B = 25$ is typical; 50 reduces variance further. Each fold is a
  full fit, so cost scales linearly in $B$.
- **K-fold**: $B = 5$ or $10$ is standard (5-fold or 10-fold CV). Larger $B$
  gives more training data per fold but fewer OOB points per fold.
- **Subsampling**: $B = 25$ with `prob=0.5` (half in-bag, half out-of-bag).
  Use it when the in-bag sample size should be fixed.

### Flat or monotone risk curves

If the cross-validated risk monotonically decreases up to $m_{\max}$, the model
has not overfit yet: refit with a larger `mstop`. If the curve is nearly flat
across all iterations:

1. **The model is correctly specified but weak.** Try increasing `df` for smooth
   terms (e.g. `df=6` instead of `df=4`) or adding more base learners.
2. **The signal-to-noise ratio is low.** Cross-validated risk near
   the null model (offset only) means there is little to learn from the data.
3. **The family is wrong.** A quantile model at $\tau=0.1$ on symmetric data may
   show a flat risk because the asymmetric loss is dominated by the symmetric
   noise.

### High-variance risk curves

With small datasets or binary outcomes, the folds may disagree substantially.
Remedies:

- Repeat the split with another seed, or increase $B$, to measure how much the
  selected $m_{\text{stop}}$ depends on the split.
- Use `kfold` instead of `bootstrap` for less optimistic bias.
- Accept the imprecision and pick $m_{\text{stop}}$ conservatively (the "one
  standard error rule": choose the simplest model within 1 SE of the minimum).

## Weights

Observation weights $w_i$ enter the risk $\sum_i w_i\,\rho(y_i, f_i)$ and the
per-observation squared error used for base learner selection. They are useful
for:

- **Case-control oversampling**: assign weight 2 to minority-class observations,
  weight 1 to majority.
- **Frequency weights**: integer weights representing duplicate observations.
- **Variance weights**: if observation $i$ has known variance $\sigma_i^2$, set
  $w_i = 1/\sigma_i^2$.
- **Survey weights**: design weights for complex sampling.

In `cvrisk`, each fold fits with its in-bag weights and evaluates the
out-of-bag risk with the original observation weights on the out-of-bag rows
(the $v^{(b)}$ of step 3), as R mboost does.

### Weight dependence of `df`

The `df` to `λ` mapping depends on $W$. When `cvrisk` reweights observations per
fold (zeroing out-of-bag weights), `λ` is recomputed so the effective `df`
stays constant, as in R mboost. A term
with `df=4` has exactly 4 effective degrees of freedom in every fold, regardless
of the fold's effective sample size. See [](penalties_df.ipynb) for the mechanics.

## Offsets

The `offset=` argument to `glmboost`/`gamboost` sets the initial predictor
$\hat f^{[0]}$ to a known value instead of the family's default offset. Use it
to:

- **Warm-start** from a previous fit: `offset=fit.fitted_` starts from the
  predictor of an existing model.
- **Fix a known intercept**: `offset=0.5 * np.log(p / (1 - p))` for a Binomial
  model with a known baseline rate `p`. The Binomial predictor is half the
  log-odds, $p = 1/(1 + e^{-2f})$, so the offset is on that scale.
- **Include an external covariate**: `offset=external_prediction` adds a
  known linear predictor before boosting starts.

If `offset=` is not provided, the family computes it automatically (e.g.,
weighted mean for Gaussian, half the log-odds of the event rate for Binomial).

## Information criteria

For Gaussian and likelihood families, an AIC-type criterion
{cite}`akaike1974new` selects $m_{\text{stop}}$ from one fit, without
resampling. The general form is a penalized log-likelihood:

$$
\mathrm{AIC}(m) = -2\,\ell\big(\hat f^{[m]}\big) + 2\,\mathrm{df}(m),
$$

where $\ell$ is the log-likelihood (evaluated at the fitted predictor) and
$\mathrm{df}(m) = \operatorname{tr}(B_m)$ is the trace of the boosting hat
matrix. Under squared-error loss, $B_m = I - \prod_{k=1}^{m}(I - \nu H_{s_k})$
is the $n \times n$ matrix that maps the response to the fitted values after
$m$ iterations, where $H_{s_k}$ is the hat matrix of the learner selected at
iteration $k$; for Binomial and Poisson each step is weighted by the working
(IRLS) weights at the current fit, as in R mboost. The penalty
$2\,\mathrm{df}(m)$ approximates the expected optimism of the training
log-likelihood as an estimate of the test log-likelihood, under the assumption
that the hat-matrix approximation is accurate.

The quantity $\operatorname{tr}(2B_m - B_m B_m^\top)$ is a different
degrees-of-freedom definition, which equals $\operatorname{tr}(B_m)$ only when
$B_m$ is symmetric and idempotent. The AIC computation does not use it;
mboostrs uses that form for a single learner, $\operatorname{tr}(2H -
HH^\top)$, to convert a learner's `df` into its penalty $\lambda$
([](penalties_df.ipynb)). [](theory_properties.ipynb) states the same AIC
definition.

Both `glmboost` and `gamboost` support the `.aic()` method:

```python
aic = fit.aic()                 # works for both GLMBoostModel and GAMBoostModel
print(f"Optimal mstop: {aic.mstop}")
mb.plot.plot_aic(aic)
fit_truncated = fit[:aic.mstop]
```


For `glmboost`, the hat-matrix trace is computed from the linear base-learners;
for `gamboost`, it uses the hat matrices of the spline and other smooth
base-learners, and for a tree learner (`btree`) the averaging matrix of the
partition fitted at each step. `aic()` raises an error for a model with a
monotone learner (`bmono`), whose constrained fit is not a linear map of the
response and so has no hat matrix, for Multinomial models, whose learners fit
all classes jointly and have no hat matrix in mboostrs, and for survival
families.

### The corrected Gaussian stopping criterion

`aic(method="corrected")` (Gaussian family only) computes R mboost's
corrected Akaike information criterion (AICc) {cite}`hurvich1989regression`:

$$
C(m) = \log\left(\frac{\mathrm{RSS}(m)}{n}\right) +
       \frac{1 + \mathrm{df}(m)/n}{1 - (\mathrm{df}(m)+2)/n}.
$$

$C(m)$ is a per-observation quantity: the logarithm of the mean squared
residual plus a penalty that grows with $\mathrm{df}(m)/n$. It ranks the
iterations of one fit. `AicResult` labels it with criterion `mboost_corrected`
and normalization `per_observation`. `aic(method="classical")` computes the
total-scale form $2 \times \text{risk} + k\,\mathrm{df}(m)$ (default
$k = 2$) for likelihood families other than Gaussian, whose risk is the
negative log-likelihood. In both forms the trace treats the sequence of
selected learners as fixed: it does not count the degrees of freedom used in
choosing which learner to update at each step, so it understates the
complexity of the selected model.

### gMDL: generalized minimum description length

For Gaussian families, an alternative criterion is the gMDL {cite}`hansen2001model`
(generalized minimum description length), which does not require an estimate of
the error variance $\sigma^2$. It is derived from a universal prior on the
signal-to-noise ratio and takes the form

$$
s_m = \frac{\mathrm{RSS}(m)}{n-\mathrm{df}(m)}, \qquad
\mathrm{gMDL}(m) = \log(s_m) + \frac{\mathrm{df}(m)}{n}
\log\left(\frac{\sum_i y_i^2-\mathrm{RSS}(m)}{\mathrm{df}(m)s_m}\right).
$$

The response sum of squares $\sum_i y_i^2$ is raw and unweighted, as in R
mboost; RSS and $n$ use the fit's observation weights. Pass `method="gmdl"`
(Gaussian family only).

### AIC compared with cross-validation

| Criterion | Pros | Cons |
|---|---|---|
| `cvrisk` | Works for every family; direct estimate of test risk | Computationally expensive ($B$ full refits) |
| AIC | No resampling; fast; deterministic | Requires a Gaussian or likelihood family; df approximation may be poor for small $n$; unavailable for `bmono`, Multinomial and survival models |

**Recommendation:** Use `cvrisk` as the default. Use AIC when resampling is
too expensive (large $n$, many base learners), or as a fast
cross-check. If AIC and `cvrisk` disagree substantially, trust `cvrisk`.



## Verifying the criteria: Friedman #1

`cvrisk`, AIC and the true (oracle) test-risk minimizer should select values
of $m_{\text{stop}}$ close to one another. The check uses Friedman's
nonlinear regression benchmark {cite}`friedman1991multivariate`:
$y = 10\sin(\pi x_0 x_1) + 20(x_2 - 0.5)^2 + 10x_3 + 5x_4 + \varepsilon$
with $\varepsilon \sim N(0, 1)$, plus five pure-noise predictors $x_5,\dots,x_9$.
One training sample of $n = 400$ is fitted with a P-spline (`bbs`, `df=4`) for
each of the ten predictors, $\nu = 0.1$ and $m_{\max} = 400$. The oracle is
the mean squared error against the *noiseless* signal on 4000 independent test
points, evaluated every 5 iterations; `cvrisk` and AIC never see it. `cvrisk`
uses 10-fold cross-validation and AIC the corrected Gaussian criterion.

In [1]:
import warnings
import numpy as np
import mboostrs as mb

warnings.filterwarnings("ignore", message="Some 'x' values are beyond")

# friedman1 returns the noiseless signal alongside y, which the oracle needs.
n_train, n_test = 400, 4000
df_train = mb.datasets.friedman1(n=n_train, n_features=10, noise=1.0, seed=0)
df_test = mb.datasets.friedman1(n=n_test, n_features=10, noise=1.0, seed=1)
cols = [f"x{i}" for i in range(10)]
data_train = {c: df_train[c].to_numpy() for c in cols} | {"y": df_train["y"].to_numpy()}
data_test = {c: df_test[c].to_numpy() for c in cols}
y_test_signal = df_test["y_signal"].to_numpy()

formula = "y ~ " + " + ".join(f"bbs({c}, df=4)" for c in cols)
mstop_max = 400
fit = mb.gamboost(formula, data=data_train, family=mb.Gaussian(),
                  control=mb.boost_control(mstop=mstop_max, nu=0.1))

# Oracle: squared error against the noiseless signal on the test set, every 5 steps.
grid = np.arange(0, mstop_max + 1, 5)
oracle_risk = np.array([
    np.mean(((np.full(n_test, fit.offset_) if m == 0
              else fit[:m].predict(newdata=data_test, type="response")) - y_test_signal) ** 2)
    for m in grid
])

In [2]:
m_oracle = int(grid[np.argmin(oracle_risk)])
cvr = fit.cvrisk(type="kfold", B=10, seed=0)
aic = fit.aic()

print(f"oracle argmin mstop  = {m_oracle}")
print(f"cvrisk optimal mstop = {cvr.optimal_mstop}")
print(f"AIC optimal mstop    = {aic.mstop}")

# The oracle minimum is an interior point: the oracle risk has turned back up by mstop_max.
assert oracle_risk[-1] > oracle_risk.min()
assert 20 < m_oracle < mstop_max - 20
# cvrisk and AIC select values within 100 iterations of the oracle.
assert abs(cvr.optimal_mstop - m_oracle) < 100
assert abs(aic.mstop - m_oracle) < 100

oracle argmin mstop  = 220
cvrisk optimal mstop = 167
AIC optimal mstop    = 190


The oracle test-risk minimum is at $m_{\text{stop}} = 220$; `cvrisk`
(10-fold) selects 167 and the corrected Gaussian criterion selects 190. Both
are within 60 iterations of the oracle and well short of $m_{\max}=400$, where
the oracle risk has already turned back up. AIC needs one fit instead of 10
and is closer to the oracle in this sample (30 against 53 iterations). This is
a single training sample, so it does not show which criterion is closer in
general; `cvrisk` remains the default because it applies to every family, and
AIC serves as a fast check.

## Post-selection inference

Boosting performs variable selection at every step. The coefficients, p-values,
and confidence intervals produced by `coef()` and `confint()` are *conditional on
the selected model* — they do not account for the selection uncertainty. This
means:

- Standard errors are too small (they condition on the selected set).
- Confidence intervals undercover (they are too narrow).
- P-values are anti-conservative.

Two approaches address this:

1. **Stability selection** ([](stability_selection.ipynb)): provides finite-sample
   error control on the set of selected variables, giving formal guarantees
   about false positives.
2. **Selective inference**: adjusts confidence intervals for the selection event.
   mboostrs does not implement selective inference; use `confint()` as a
   descriptive summary.

## Distributional and multi-parameter models

For `gamboostLSS` (location, scale and shape models: each parameter of the response distribution has its own additive predictor), $m_{\text{stop}}$ is a vector, one per distribution parameter, under cyclic fitting (every parameter is updated once per step); `cvrisk_glmboostLSS` searches the joint grid using the
joint out-of-bag negative log-likelihood. Non-cyclic fitting reduces this to a
single $m_{\text{stop}}$. See [](gamlss.ipynb) for details.

## Practical guidance

- Fix the step length $\nu$ at a small value (0.1) and tune only $m_{\text{stop}}$.
- Use enough $m_{\max}$ that the cross-validated risk curve clearly turns up; if
  the minimum is at the boundary, increase $m_{\max}$.
- Choose the fold count for the resampling design and compute budget. More
  folds do not guarantee a lower-variance selected $\hat m$ because their
  training sets overlap more strongly.
- Selection is *post hoc*: inference conditional on $\hat m$ understates
  uncertainty (see [](stability_selection.ipynb) for a procedure with formal error
  control).
- If the risk curve is noisy, increase $B$ or use `kfold`. If it never
  turns up, increase `mstop` or add richer learners.

## Summary

- Cross-validation estimates held-out loss for a specified resampling design.
  The corrected Gaussian and gMDL criteria are cheaper conditional-path
  approximations with different units and assumptions.
- The criteria use the trace of the boosting hat matrix, $\operatorname{tr}(B_m)$,
  with the selected learners treated as fixed. It omits the degrees of freedom
  spent choosing learners, so it understates the complexity of the selected
  model.
- The three risk-recording options (inbag, oobag, and none) differ in what is
  recorded during fitting. Cross-validation always uses out-of-bag risk,
  independent of the fitting risk.

## See also

- [](functional_gradient_boosting.ipynb) — the $B_m$ hat matrix and df derivation.
- [](stability_selection.ipynb) — variable selection with formal error control
  (complementary to CV-based stopping).
- [](penalties_df.ipynb) — the per-learner df-to-$\lambda$ mapping.
- [](../usage/model_selection) — runnable examples of cvrisk, AIC, and gMDL.